# 05 — Screen DB-COMP documents with Gemma 31B

## Purpose

This notebook screens the processed DB-COMP documents created by
`04_process_dbcomp.ipynb` and identifies which files belong in the substantive
European Commission antitrust corpus.

The task is deliberately narrow:

> **What kind of document is this, and should it be retained as a substantive Commission antitrust decision?**

The classifier looks at the **document itself**. A press release, summary notice or
other document that describes a Commission decision is not treated as the decision
simply because it refers to one.

The main distinction is between:

- substantive Commission antitrust decisions that are relevant to the dataset;
- rejection-of-complaint decisions, which are identified separately;
- press releases and similar communications;
- a small number of other surrounding or non-substantive document types.

This notebook is a **screening step**, not a substantive legal analysis of retained
Commission decisions. It does not infer decision types, legal outcomes, remedies or
appeal relationships.

## Inputs and outputs

The input is the compact text-processing manifest from notebook 04:

`output/com_dbcomp/db_comp_text_manifest.csv`

The classifier reads the canonical text files under:

`data/processed/com_dbcomp/text/`

Classification progress is written after each completed document to:

`logs/com_dbcomp/db_comp_classification_state.jsonl`

This JSONL file is an operational checkpoint that allows an interrupted run to resume.
It is not the publication-facing dataset table.

The final screening results are written to:

`output/com_dbcomp/db_comp_classification.csv`

Paths stored in outputs remain portable and do not contain usernames, drive letters
or machine-specific parent directories.

In [1]:
from __future__ import annotations

import asyncio
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import httpx
import pandas as pd
from tqdm.auto import tqdm


REPOSITORY_LABEL = "ec_cjeu"


def find_project_root(start: Path | None = None) -> Path:
    """Locate the repository root from the code/01_commission layout."""
    override = os.environ.get("ECCJEU_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"ECCJEU_PROJECT_ROOT does not exist: {root}")
        return root

    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "code" / "01_commission").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook from within the "
        "ec_cjeu repository or set ECCJEU_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root()

INPUT_MANIFEST = PROJECT_ROOT / "output" / "com_dbcomp" / "db_comp_text_manifest.csv"
LOG_DIR = PROJECT_ROOT / "logs" / "com_dbcomp"
OUTPUT_DIR = PROJECT_ROOT / "output" / "com_dbcomp"

CLASSIFICATION_STATE_JSONL = LOG_DIR / "db_comp_classification_state.jsonl"
CLASSIFICATION_CSV = OUTPUT_DIR / "db_comp_classification.csv"

for directory in (LOG_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def resolve_repo_path(value: str) -> Path:
    """Resolve data/... or ec_cjeu/data/... paths stored by earlier notebooks."""
    value = str(value or "").strip().replace("\\", "/")
    if not value:
        raise ValueError("Empty repository path.")

    parts = Path(value).parts
    if parts and parts[0] == REPOSITORY_LABEL:
        value = Path(*parts[1:]).as_posix()

    path = Path(value)
    if path.is_absolute() or ".." in path.parts or re.match(r"^[A-Za-z]:[/\\]", value):
        raise ValueError(f"Non-portable path: {value}")

    return PROJECT_ROOT / path


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


print("Input:  output/com_dbcomp/db_comp_text_manifest.csv")
print("Log:    logs/com_dbcomp/db_comp_classification_state.jsonl")
print("Output: output/com_dbcomp/db_comp_classification.csv")

Input:  output/com_dbcomp/db_comp_text_manifest.csv
Log:    logs/com_dbcomp/db_comp_classification_state.jsonl
Output: output/com_dbcomp/db_comp_classification.csv


## 1. Classification configuration

Gemma 31B is served locally through an OpenAI-compatible vLLM endpoint.

The classification is deterministic (`temperature = 0`) and the model returns a short
structured JSON object. Parallel requests are used only to improve throughput; each
document remains an independent classification job.

`CLASSIFICATION_VERSION` should be increased whenever the prompt, taxonomy or document
context rule is changed. Existing JSONL results are reused only when the text checksum,
model and classification version still match.

In [2]:
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "EMPTY")
MODEL = os.getenv("VLLM_MODEL", "google/gemma-4-31B-it")

CLASSIFICATION_VERSION = "1.0"

TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 300
REQUEST_TIMEOUT_SECONDS = int(os.getenv("REQUEST_TIMEOUT_SECONDS", "3600"))
CONCURRENT_REQUESTS = int(os.getenv("CONCURRENT_REQUESTS", "6"))
MAX_RETRIES = int(os.getenv("MAX_RETRIES", "1"))

FORCE_RECLASSIFY = False
TEST_MODE = False
TEST_LIMIT = 10

MAX_FULL_TEXT_CHARS = 120_000
HEAD_CHARS = 50_000
TAIL_CHARS = 50_000

print("Model:", MODEL)
print("Classification version:", CLASSIFICATION_VERSION)
print("Concurrent requests:", CONCURRENT_REQUESTS)
print("Maximum output tokens:", MAX_OUTPUT_TOKENS)

Model: google/gemma-4-31B-it
Classification version: 1.0
Concurrent requests: 6
Maximum output tokens: 300


## 2. Load the processed DB-COMP collection

Notebook 04 already links every DB-COMP source document to one canonical processed text
file. This notebook therefore does not merge legacy manifests or search through several
possible text-path columns.

Only rows with `processing_status == "success"` are classified. The text file must exist
and its stored SHA-256 checksum is carried forward as the identity of the exact text
classified by Gemma.

In [3]:
REQUIRED_COLUMNS = {
    "source",
    "dbcomp_document_id",
    "case_number",
    "case_number_source",
    "case_type",
    "dbcomp_date",
    "document_name",
    "file_url",
    "file_name",
    "raw_path",
    "raw_sha256",
    "text_path",
    "text_sha256",
    "processing_status",
}


def read_manifest(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {path}")
    return pd.read_csv(path, dtype=str, keep_default_na=False)


manifest = read_manifest(INPUT_MANIFEST)

missing_columns = sorted(REQUIRED_COLUMNS - set(manifest.columns))
if missing_columns:
    raise ValueError(f"Input manifest is missing required columns: {missing_columns}")

if manifest["file_url"].eq("").any() or not manifest["file_url"].is_unique:
    raise ValueError("file_url must be unique and non-empty.")

successful = manifest["processing_status"].str.lower().eq("success")
work = manifest.loc[successful].copy()

if len(work) != len(manifest):
    print(f"Skipping {len(manifest) - len(work):,} row(s) not successfully processed.")

text_paths = [resolve_repo_path(value) for value in work["text_path"]]

missing_text = [
    work.iloc[index]["file_name"]
    for index, path in enumerate(text_paths)
    if not path.exists() or not path.is_file() or path.stat().st_size == 0
]
if missing_text:
    raise FileNotFoundError(
        f"{len(missing_text)} processed text file(s) are missing; "
        f"examples: {missing_text[:10]}"
    )

work["text_path_internal"] = [str(path) for path in text_paths]
work = work.reset_index(drop=True)

if TEST_MODE:
    work = work.head(TEST_LIMIT).copy()

print(f"Documents available for classification: {len(work):,}")
display(
    work[
        [
            "dbcomp_document_id",
            "case_number",
            "document_name",
            "file_name",
            "text_path",
        ]
    ].head()
)

Documents available for classification: 837


,dbcomp_document_id,case_number,document_name,file_name,text_path
0,,39904,Rechargeable batteries,AT_39904_5942450_1110_7.pdf,ec_cjeu/data/processed/com_dbcomp/text/AT_3990...
1,,40135,FOREX - Three Way Banana Split,AT_40135_8399795_5456_9.pdf,ec_cjeu/data/processed/com_dbcomp/text/AT_4013...
2,,40136,Capacitors,AT_40136_7970992_6095_3.pdf,ec_cjeu/data/processed/com_dbcomp/text/AT_4013...
3,,40490,Spanish Football Trainers,AT_40490_8263501_702_3.pdf,ec_cjeu/data/processed/com_dbcomp/text/AT_4049...
4,,40522,Metal packaging,AT_40522_8986595_2868_7.pdf,ec_cjeu/data/processed/com_dbcomp/text/AT_4052...


## 3. Screening taxonomy

The taxonomy is intentionally small and follows the actual purpose of this step.

- **`relevant`** — the document itself is a substantive operative European Commission
  antitrust decision under Articles 101, 102 or 106 TFEU, or their historical predecessors.
- **`rejection_of_complaint`** — a Commission decision rejecting a complaint. These are
  identified separately and are not retained in the main corpus.
- **`press_release_or_memo`** — a press release, MEMO, factsheet or similar communication
  about a case or decision.
- **`market_test_notice`** — an Article 27(4) or similar market-test notice rather than the
  final decision.
- **`summary_or_publication_notice`** — an Official Journal notice, summary or other
  publication that reports a decision but is not the decision itself.
- **`other_non_substantive`** — another surrounding, administrative or non-operative
  document rather than the substantive Commission decision.
- **`unclear`** — the supplied text does not provide enough evidence for a reliable choice.

The final Boolean `include` is derived directly from this classification:
only `document_class == "relevant"` is retained.

In [4]:
DOCUMENT_CLASSES = [
    "relevant",
    "rejection_of_complaint",
    "press_release_or_memo",
    "market_test_notice",
    "summary_or_publication_notice",
    "other_non_substantive",
    "unclear",
]

RESPONSE_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "document_class": {
            "type": "string",
            "enum": DOCUMENT_CLASSES,
        },
        "reason": {
            "type": "string",
        },
        "evidence": {
            "type": "string",
        },
    },
    "required": [
        "document_class",
        "reason",
        "evidence",
    ],
    "additionalProperties": False,
}


SYSTEM_PROMPT = """
You are helping screen documents collected from DB-COMP for a research dataset of
substantive European Commission antitrust decisions.

Read the supplied document and classify THE DOCUMENT ITSELF. Do not call it a Commission
decision simply because it mentions, announces, summarizes or quotes one.

Use exactly one label:

relevant
- The document itself is a substantive operative European Commission antitrust decision
  under Articles 101, 102 or 106 TFEU, or their historical predecessors.

rejection_of_complaint
- The document itself is a Commission decision rejecting a complaint. These are identified
  separately and are not retained in the main corpus.

press_release_or_memo
- A press release, MEMO, factsheet or similar communication about a case or decision.

market_test_notice
- An Article 27(4) or similar market-test notice rather than the final Commission decision.

summary_or_publication_notice
- An Official Journal notice, summary or similar publication that reports a decision but
  is not the operative decision itself.

other_non_substantive
- Another surrounding, administrative or non-operative document rather than the substantive
  Commission decision.

unclear
- The supplied text does not contain enough evidence to decide reliably.

Important distinctions:
- a press release announcing a Commission decision is press_release_or_memo;
- the actual operative Commission decision is relevant;
- a market-test notice is market_test_notice;
- the later final commitment decision is relevant;
- a rejection of complaint is rejection_of_complaint even though it is formally a decision.

Return:
- document_class: one label from the list above;
- reason: one short sentence explaining the classification;
- evidence: one short decisive phrase from the supplied document.

Keep the answer concise. Return only the structured JSON requested by the schema.
""".strip()

## 4. Document context sent to Gemma

For documents up to `MAX_FULL_TEXT_CHARS`, Gemma receives the complete processed text.

For longer documents, the first `HEAD_CHARS` and last `TAIL_CHARS` characters are supplied,
with the middle omitted. This keeps the classification task manageable while preserving the
parts of a Commission document most useful for identifying document type: headings and
introductory language near the beginning, and operative provisions or signatures near the end.

The processing manifest already provides source metadata, so the prompt contains only a small
amount of identifying context in addition to the document text.

In [5]:
def read_text_for_llm(path: Path) -> tuple[str, str]:
    text = path.read_text(encoding="utf-8", errors="replace")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    if len(text) <= MAX_FULL_TEXT_CHARS:
        return text, "full"

    selected = (
        text[:HEAD_CHARS]
        + "\n\n[... middle omitted for classification ...]\n\n"
        + text[-TAIL_CHARS:]
    )
    return selected, "head_tail"


def build_user_prompt(row: pd.Series) -> tuple[str, str]:
    text_path = Path(row["text_path_internal"])
    text, text_selection = read_text_for_llm(text_path)

    metadata = {
        "dbcomp_document_id": row.get("dbcomp_document_id", ""),
        "case_number": row.get("case_number", ""),
        "document_name": row.get("document_name", ""),
        "file_name": row.get("file_name", ""),
        "dbcomp_date": row.get("dbcomp_date", ""),
    }

    prompt = (
        "Source metadata:\n"
        + json.dumps(metadata, ensure_ascii=False)
        + "\n\nDocument text:\n"
        + text
    )

    return prompt, text_selection

## 5. Validate structured model responses

The model is not asked for a numerical confidence score. The screening decision is simply
the returned document class.

`unclear` is available when the text does not support a reliable classification. This is
more interpretable than applying bespoke confidence thresholds to an uncalibrated model score.

In [6]:
def extract_json_object(text: str) -> dict[str, Any]:
    text = (text or "").strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        return json.loads(text[start:end + 1])

    raise ValueError(f"Could not parse JSON response: {text[:500]}")


def validate_model_response(obj: dict[str, Any]) -> dict[str, str]:
    document_class = str(obj.get("document_class", "")).strip().lower()

    if document_class not in DOCUMENT_CLASSES:
        document_class = "unclear"

    reason = str(obj.get("reason", "") or "").strip()
    evidence = str(obj.get("evidence", "") or "").strip()

    return {
        "document_class": document_class,
        "reason": reason[:400],
        "evidence": evidence[:300],
    }

## 6. Resumable JSONL state

The JSONL file is used only to resume classification and troubleshoot failed calls.

Each completed classification stores the source URL, processed-text checksum, model,
classification version, text-selection rule and compact classification result.

A previous result is reused only when:

- the model is unchanged;
- the classification version is unchanged;
- the processed text SHA-256 is unchanged; and
- the earlier call completed without an error.

Failed calls are retried on the next run. Set `FORCE_RECLASSIFY = True` to deliberately
classify every document again.

In [7]:
def load_classification_state() -> dict[str, dict[str, Any]]:
    latest: dict[str, dict[str, Any]] = {}

    if not CLASSIFICATION_STATE_JSONL.exists():
        return latest

    with CLASSIFICATION_STATE_JSONL.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            file_url = str(record.get("file_url", "")).strip()
            if file_url:
                latest[file_url] = record

    return latest


def append_classification_state(record: dict[str, Any]) -> None:
    CLASSIFICATION_STATE_JSONL.parent.mkdir(parents=True, exist_ok=True)

    if CLASSIFICATION_STATE_JSONL.exists() and CLASSIFICATION_STATE_JSONL.stat().st_size > 0:
        with CLASSIFICATION_STATE_JSONL.open("rb") as handle:
            handle.seek(-1, os.SEEK_END)
            last_byte = handle.read(1)

        if last_byte != b"\n":
            with CLASSIFICATION_STATE_JSONL.open("ab") as handle:
                handle.write(b"\n")
                handle.flush()
                os.fsync(handle.fileno())

    payload = (json.dumps(record, ensure_ascii=False, default=str) + "\n").encode("utf-8")

    with CLASSIFICATION_STATE_JSONL.open("ab") as handle:
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())


def state_is_reusable(record: dict[str, Any] | None, row: pd.Series) -> bool:
    if FORCE_RECLASSIFY or not record:
        return False

    if record.get("model") != MODEL:
        return False

    if record.get("classification_version") != CLASSIFICATION_VERSION:
        return False

    if record.get("text_sha256") != row.get("text_sha256"):
        return False

    if str(record.get("error", "")).strip():
        return False

    if record.get("document_class") not in DOCUMENT_CLASSES:
        return False

    return True


classification_state = load_classification_state()

todo_mask = [
    not state_is_reusable(
        classification_state.get(row["file_url"]),
        row,
    )
    for _, row in work.iterrows()
]

todo = work.loc[todo_mask].copy()

print(f"Existing reusable classifications: {len(work) - len(todo):,}")
print(f"Documents to classify: {len(todo):,}")

Existing reusable classifications: 145
Documents to classify: 692


## 7. Gemma 31B inference

Each document is an independent request to the local vLLM endpoint. A semaphore limits the
number of simultaneous requests while vLLM handles batching internally.

`tqdm` provides the live progress bar. Only failed/retried requests are printed separately;
successful classifications are written directly to the JSONL checkpoint.

In [8]:
async def call_vllm(
    client: httpx.AsyncClient,
    user_prompt: str,
) -> tuple[str, str]:
    endpoint = f"{VLLM_BASE_URL.rstrip('/')}/chat/completions"

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_OUTPUT_TOKENS,
        "structured_outputs": {
            "json": RESPONSE_JSON_SCHEMA,
        },
    }

    response = await client.post(
        endpoint,
        json=payload,
        headers={"Authorization": f"Bearer {VLLM_API_KEY}"},
    )
    response.raise_for_status()

    body = response.json()
    choices = body.get("choices") or []
    if not choices:
        raise ValueError("vLLM response contains no choices.")

    choice = choices[0]
    content = choice.get("message", {}).get("content")

    if not content:
        raise ValueError("vLLM response contains no message content.")

    return str(content), str(choice.get("finish_reason") or "")


async def classify_one(
    row_dict: dict[str, Any],
    client: httpx.AsyncClient,
    semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    row = pd.Series(row_dict)
    prompt, text_selection = build_user_prompt(row)

    last_error = ""

    for attempt in range(MAX_RETRIES + 1):
        try:
            async with semaphore:
                content, finish_reason = await call_vllm(client, prompt)

            if finish_reason == "length":
                raise RuntimeError(
                    f"Model response reached MAX_OUTPUT_TOKENS={MAX_OUTPUT_TOKENS}"
                )

            parsed = extract_json_object(content)
            classification = validate_model_response(parsed)

            return {
                "file_url": row["file_url"],
                "text_sha256": row["text_sha256"],
                "classification_version": CLASSIFICATION_VERSION,
                "model": MODEL,
                "text_selection": text_selection,
                "document_class": classification["document_class"],
                "reason": classification["reason"],
                "evidence": classification["evidence"],
                "classified_at_utc": utc_now_iso(),
                "error": "",
            }

        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"

            if attempt < MAX_RETRIES:
                tqdm.write(
                    f"Retry | {row.get('file_name', '')} | "
                    f"attempt {attempt + 1}/{MAX_RETRIES + 1} | {last_error}"
                )
                await asyncio.sleep(2 ** attempt)

    tqdm.write(f"Failed | {row.get('file_name', '')} | {last_error}")

    return {
        "file_url": row["file_url"],
        "text_sha256": row["text_sha256"],
        "classification_version": CLASSIFICATION_VERSION,
        "model": MODEL,
        "text_selection": "",
        "document_class": "unclear",
        "reason": "",
        "evidence": "",
        "classified_at_utc": utc_now_iso(),
        "error": last_error,
    }


async def run_classification(todo_df: pd.DataFrame) -> dict[str, dict[str, Any]]:
    latest = load_classification_state()

    if todo_df.empty:
        print("No new classifications required.")
        return latest

    timeout = httpx.Timeout(REQUEST_TIMEOUT_SECONDS)
    limits = httpx.Limits(
        max_connections=max(CONCURRENT_REQUESTS * 2, 8),
        max_keepalive_connections=max(CONCURRENT_REQUESTS, 4),
    )
    semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)

    rows = todo_df.to_dict("records")

    counts = {
        "relevant": 0,
        "rejection": 0,
        "press": 0,
        "other": 0,
        "unclear": 0,
        "failed": 0,
    }

    async with httpx.AsyncClient(timeout=timeout, limits=limits) as client:
        tasks = [
            asyncio.create_task(classify_one(row, client, semaphore))
            for row in rows
        ]

        progress = tqdm(
            asyncio.as_completed(tasks),
            total=len(tasks),
            desc="Gemma DB-COMP screening",
            unit="doc",
            dynamic_ncols=True,
        )

        for completed in progress:
            record = await completed

            append_classification_state(record)
            latest[record["file_url"]] = record

            if record["error"]:
                counts["failed"] += 1
            elif record["document_class"] == "relevant":
                counts["relevant"] += 1
            elif record["document_class"] == "rejection_of_complaint":
                counts["rejection"] += 1
            elif record["document_class"] == "press_release_or_memo":
                counts["press"] += 1
            elif record["document_class"] == "unclear":
                counts["unclear"] += 1
            else:
                counts["other"] += 1

            progress.set_postfix(
                {
                    "rel": counts["relevant"],
                    "reject": counts["rejection"],
                    "press": counts["press"],
                    "other": counts["other"],
                    "unclear": counts["unclear"],
                    "fail": counts["failed"],
                },
                refresh=False,
            )

    return latest

## 8. Run classification

In [9]:
latest_results = await run_classification(todo)

print(f"Classification records available: {len(latest_results):,}")
print("Checkpoint: logs/com_dbcomp/db_comp_classification_state.jsonl")

Gemma DB-COMP screening:   0%|          | 0/692 [00:00<?, ?doc/s]

Classification records available: 837
Checkpoint: logs/com_dbcomp/db_comp_classification_state.jsonl


## 9. Build the compact classification manifest

The final CSV keeps the DB-COMP source metadata, the raw and processed document paths,
the processed-text checksum and the compact screening result.

The JSONL checkpoint retains operational details needed for resuming the model run; these
do not need to be duplicated as a large audit table in the publication-facing CSV.

In [10]:
latest_results = load_classification_state()

rows = []

for _, source_row in manifest.iterrows():
    result = latest_results.get(source_row["file_url"], {})

    document_class = result.get("document_class", "")
    error = result.get("error", "")

    rows.append(
        {
            "source": source_row["source"],
            "dbcomp_document_id": source_row["dbcomp_document_id"],
            "case_number": source_row["case_number"],
            "case_number_source": source_row["case_number_source"],
            "case_type": source_row["case_type"],
            "dbcomp_date": source_row["dbcomp_date"],
            "document_name": source_row["document_name"],
            "file_url": source_row["file_url"],
            "file_name": source_row["file_name"],
            "raw_path": source_row["raw_path"],
            "text_path": source_row["text_path"],
            "text_sha256": source_row["text_sha256"],
            "document_class": document_class,
            "include": document_class == "relevant" and not bool(error),
            "classification_reason": result.get("reason", ""),
            "classification_evidence": result.get("evidence", ""),
            "text_selection": result.get("text_selection", ""),
            "model": result.get("model", ""),
            "classification_version": result.get("classification_version", ""),
            "classification_error": error,
        }
    )

classification_manifest = pd.DataFrame(rows)

classification_manifest.to_csv(
    CLASSIFICATION_CSV,
    index=False,
    encoding="utf-8",
)

print("Saved: output/com_dbcomp/db_comp_classification.csv")
display(
    classification_manifest[
        [
            "dbcomp_document_id",
            "case_number",
            "document_name",
            "document_class",
            "include",
            "classification_reason",
        ]
    ].head()
)

Saved: output/com_dbcomp/db_comp_classification.csv


,dbcomp_document_id,case_number,document_name,document_class,include,classification_reason
0,,39904,Rechargeable batteries,relevant,True,The document is a substantive operative Commis...
1,,40135,FOREX - Three Way Banana Split,relevant,True,The document is a substantive operative Commis...
2,,40136,Capacitors,relevant,True,The document is a substantive operative Commis...
3,,40490,Spanish Football Trainers,rejection_of_complaint,False,The document is a formal decision by the Commi...
4,,40522,Metal packaging,relevant,True,The document is a substantive operative Commis...


## 10. Classification summary and validation

The summary below is intentionally concise. The model output is used as a screening aid,
not as an independent measure of legal truth.

Before the final Scientific Data release, the semantic screening should be validated
against human inspection. A practical validation design is to inspect:

- every `unclear` or failed classification;
- all rare exclusion classes;
- a random sample of `relevant` documents;
- a random sample of the two common exclusion classes
  (`rejection_of_complaint` and `press_release_or_memo`).

The final sample size and agreement statistics should be fixed and reported with the
Data Descriptor once the complete corpus has been screened.

In [11]:
class_counts = (
    classification_manifest["document_class"]
    .replace("", "(not classified)")
    .value_counts(dropna=False)
)

print("Document classes:")
print(class_counts.to_string())

print()
print(f"Included: {int(classification_manifest['include'].sum()):,}")
print(
    "Classification errors:",
    int(classification_manifest["classification_error"].astype(str).str.strip().ne("").sum()),
)

problem_rows = classification_manifest.loc[
    classification_manifest["document_class"].isin(["unclear", ""])
    | classification_manifest["classification_error"].astype(str).str.strip().ne("")
].copy()

if len(problem_rows):
    print()
    print(f"Rows requiring inspection: {len(problem_rows):,}")
    display(
        problem_rows[
            [
                "dbcomp_document_id",
                "case_number",
                "document_name",
                "document_class",
                "classification_reason",
                "classification_error",
            ]
        ].head(20)
    )

Document classes:
document_class
relevant                         647
rejection_of_complaint           109
press_release_or_memo             62
summary_or_publication_notice     12
other_non_substantive              5
market_test_notice                 2

Included: 647
Classification errors: 0


## Reproducibility notes

The screening model used by default is `google/gemma-4-31B-it`, served locally through an
OpenAI-compatible vLLM endpoint with deterministic generation (`temperature = 0`).

The fixed taxonomy, complete prompt, structured-output schema, document-context rule and
classification version are contained directly in this notebook. The exact processed text
used for each document is identified by `text_sha256`.

Python and vLLM environment versions should be frozen at repository release level
(for example in `requirements.txt` and the documented vLLM environment) rather than
repeated in every classification row.

The JSONL checkpoint exists to make the computation resumable. The compact CSV is the
publication-facing output of this screening stage.